# Solar Filament Segmentation 2026: run 3

What changes from runs 1 and 2:

* **Extra input channels.** Besides the normalized image, the network sees the image with limb
  darkening removed, a local-contrast map and a dark-ridge map (`solarseg/features.py`).
* **Four models instead of one.** Training images are split into 4 folds by month; each model
  holds out one fold, so every training image gets an out-of-fold prediction. The two GPUs
  train two folds at a time. Weights are averaged over training steps (EMA).
* **Tuning on all 707 training images** (out-of-fold maps) instead of 106 held-out images.
* **A filament picker.** A small LightGBM model predicts how well each candidate piece will
  match the annotators and keeps a piece only when that is high enough. It is used for the
  submission only if it beats the low/high threshold rule out of fold.
* The test set is predicted by averaging the four models.

**Before running:** add the competition data (*Add Input > Competitions > Solar Filament
Segmentation Challenge 2026*), choose *Settings > Accelerator > GPU T4 x2* and turn
*Settings > Internet* on. A full run takes about 3 hours.

In [ ]:
REPO_URL = "https://github.com/FarnooshMemari/solar-filament-segmentation.git"
BRANCH = "run3"  # switch to "main" once the run 3 pull request is merged
REPO = "/tmp/solar-filament-segmentation"  # outside /kaggle/working, so it isn't saved as output

import glob, json, os, shutil, subprocess, time
TEMP = "/kaggle/temp" if os.path.isdir("/kaggle/temp") else "/tmp"  # scratch space, not saved
CACHE, PROBS = f"{TEMP}/cache", f"{TEMP}/probs"
OUT = "/kaggle/working"
RUNS, FINAL = f"{OUT}/runs", f"{OUT}/runs/run3"
FOLDS, CHANNELS = 4, "z,flat,contrast,ridge"

if not os.path.exists(REPO):
    !git clone -q -b {BRANCH} {REPO_URL} {REPO}
%cd {REPO}
!git log -1 --format="code version: %h %s"
!pip install -q pycocotools
try:
    import lightgbm
except ImportError:
    !pip install -q lightgbm
import lightgbm, torch
n_gpu = torch.cuda.device_count()
print("lightgbm", lightgbm.__version__, "| torch", torch.__version__, "| GPUs:", n_gpu)
!nvidia-smi --query-gpu=name,memory.total --format=csv
print(f"free disk in {TEMP}: {shutil.disk_usage(TEMP).free / 1e9:.0f} GB")

## 1. Compute the input channels once
Stored as float16 in the scratch folder and shared by both training processes
(about 9 GB for the training and test images).

In [ ]:
!python scripts/cache.py --out {CACHE} --channels {CHANNELS} --workers 4

## 2. Train the four fold models
Each GPU trains two folds, one after the other, and right after each fold saves probability
maps for that fold's held-out images and for the test images (8-view test-time augmentation).
Progress is printed every 5 minutes; full logs are in `logs/`.

In [ ]:
TRAIN = (f"--folds {FOLDS} --channels {CHANNELS} --cache {CACHE} --epochs 40 --steps-per-epoch 200 "
         f"--ema 0.999 --val-every 5 --tile 1024 --workers 1")
SAVE = f"--cache {CACHE} --out {PROBS} --test --tta 8 --tile 1024"
os.makedirs(f"{OUT}/logs", exist_ok=True)
gpus = list(range(max(1, n_gpu)))

def start(gpu):
    steps = []
    for k in range(gpu, FOLDS, len(gpus)):
        run = f"{RUNS}/fold{k}"
        steps.append(f"python scripts/train.py --out {run} --fold {k} {TRAIN} && "
                     f"python scripts/oof.py --run {run} {SAVE}")
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "PYTHONUNBUFFERED": "1"}
    log = open(f"{OUT}/logs/gpu{gpu}.log", "w")
    return subprocess.Popen(["bash", "-c", " && ".join(steps)], stdout=log, stderr=subprocess.STDOUT,
                            env=env, cwd=REPO)

def last_line(gpu):
    lines = [l for l in open(f"{OUT}/logs/gpu{gpu}.log").read().splitlines() if l.strip()]
    return lines[-1][:160] if lines else "(starting)"

t0 = time.time()
procs = [start(g) for g in gpus]
while any(p.poll() is None for p in procs):
    time.sleep(300)
    print(f"{(time.time() - t0) / 60:.0f} min", " | ".join(f"gpu{g}: {last_line(g)}" for g in gpus), flush=True)
for g, p in zip(gpus, procs):
    if p.returncode:
        print(open(f"{OUT}/logs/gpu{g}.log").read()[-4000:])
assert all(p.returncode == 0 for p in procs), "a training process failed; see the log above"
for k in range(FOLDS):
    rows = [json.loads(l) for l in open(f"{RUNS}/fold{k}/metrics.jsonl")]
    best = max((r for r in rows if "val_dice" in r), key=lambda r: r["val_dice"])
    print(f"fold {k}: best validation Dice {best['val_dice']:.4f} at epoch {best['epoch']}")
print(f"training and saving maps took {(time.time() - t0) / 60:.0f} min")

## 3. Tune the low/high threshold rule on all out-of-fold maps

In [ ]:
GRID = ("--low-thresholds 0.3,0.35,0.4,0.45,0.5 --thresholds 0.6,0.7,0.8,0.85,0.9,0.95,0.97 "
        "--merge-dists 0,10,25 --min-areas 150,300,600,1000 --close-radii 0 --fill-holes 0")
!python scripts/tune.py --probs {PROBS} --cache {CACHE} --out {FINAL} --workers 4 {GRID}

## 4. Train the filament picker
Out-of-fold PQ of: keeping every piece, the tuned threshold rule, the picker, and an oracle
that knows each piece's true matched IoU (an upper bound for any picker).

In [ ]:
FOLD_RUNS = " ".join(f"{RUNS}/fold{k}" for k in range(FOLDS))
!python scripts/scorer.py --probs {PROBS} --cache {CACHE} --runs {FOLD_RUNS} --post {FINAL}/postprocess.json --out {FINAL} --workers 4

## 5. Predict the test set with the four models
Writes both versions, then copies the one with the better out-of-fold PQ to `submission.csv`.

In [ ]:
oof = json.load(open(f"{FINAL}/picker.json"))["oof"]
rule_pq, picker_pq = oof["tuned_hysteresis_same_pieces"]["pq"], oof["picker"]["pq"]
COMMON = f"--probs {PROBS} --cache {CACHE} --params {FINAL}/postprocess.json --workers 4"
!python scripts/predict.py {COMMON} --out {OUT}/submission_rule.csv
!python scripts/check_submission.py --csv {OUT}/submission_rule.csv
!python scripts/predict.py {COMMON} --picker {FINAL} --out {OUT}/submission_picker.csv
!python scripts/check_submission.py --csv {OUT}/submission_picker.csv
choice = "picker" if picker_pq > rule_pq + 0.002 else "rule"
shutil.copy(f"{OUT}/submission_{choice}.csv", f"{OUT}/submission.csv")
print(f"out-of-fold PQ: threshold rule {rule_pq:.4f}, picker {picker_pq:.4f} -> submission.csv uses the {choice}")

## 6. Figures, package versions and clean-up

In [ ]:
!python scripts/visualize.py --run {RUNS}/fold0 --params {FINAL}/postprocess.json --tta 8 --n 4 --out {OUT}/figures
!pip freeze > {OUT}/requirements_frozen.txt
for k in range(FOLDS):  # keep best.pt only
    if os.path.exists(f"{RUNS}/fold{k}/last.pt"):
        os.remove(f"{RUNS}/fold{k}/last.pt")
from IPython.display import Image, display
for path in sorted(glob.glob(f"{OUT}/figures/*.png"))[:2]:
    display(Image(path))

## 7. Submit

Submit `submission.csv` from the Output tab only if both checks above printed
`submission=ok` and the out-of-fold PQ is higher than run 2's validation PQ (0.410).